![wordcloud](wordcloud.png)

As a Data Scientist working for a mobile app company, you usually find yourself applying product analytics to better understand user behavior, uncover patterns, and reveal insights to identify the great and not-so-great features. Recently, the number of negative reviews has increased on Google Play, and as a consequence, the app's rating has been decreasing. The team has requested you to analyze the situation and make sense of the negative reviews.

It's up to you to apply K-means clustering from scikit-learn and NLP techniques through NLTK to sort text data from negative reviews in the Google Play Store into categories!

## The Data

A dataset has been shared with a sample of reviews and their respective scores (from 1 to 5) in the Google Play Store. A summary and preview are provided below.

# reviews.csv

| Column     | Description              |
|------------|--------------------------|
| `'content'` | Content (text) of each review. |
| `'score'` | Score assigned to the review by the user as an integer (from 1 to 5). |

Project Instructions

To reveal the main topics from app reviews, you'll perform these tasks:

    Preprocess the negative reviews (reviews with a score of 1 or 2) by tokenizing the text, removing stop words and non-alpha characters. Save the results in a pandas DataFrame called preprocessed_reviews.
    Vectorize the cleaned negative reviews using TF-IDF and store the matrix in a variable called tfidf_matrix.
    Apply K-means clustering to tfidf_matrix to group the reviews into five categories. Store the predicted labels in a list called categories.
    For each unique cluster label, find the most frequent term. Store the results in a pandas DataFrame called topic_terms with at least three columns to store the label assigned from K-means, the identified term, and its frequency.

In [19]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [20]:
# Download necessary files from NLTK:
# punkt -> Tokenization
# stopwords -> Stop words removal
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/repl/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/repl/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
# Load the reviews dataset and preview it
reviews = pd.read_csv("reviews.csv")
reviews.head()

,content,score
0,I cannot open the app anymore,1
1,I have been begging for a refund from this app...,1
2,Very costly for the premium version (approx In...,1
3,"Used to keep me organized, but all the 2020 UP...",1
4,Dan Birthday Oct 28,1


In [22]:
negative_reviews = reviews[reviews["score"].isin([1, 2])]
negative_reviews.head()

,content,score
0,I cannot open the app anymore,1
1,I have been begging for a refund from this app...,1
2,Very costly for the premium version (approx In...,1
3,"Used to keep me organized, but all the 2020 UP...",1
4,Dan Birthday Oct 28,1


In [23]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    words = word_tokenize(text.lower())
    words = [w for w in words if w.isalpha() and w not in stop_words]
    return " ".join(words)

negative_reviews_cleaned = negative_reviews["content"].apply(clean_text)


In [24]:
preprocessed_reviews = pd.DataFrame({"review": negative_reviews_cleaned})
preprocessed_reviews.head()

,review
0,open app anymore
1,begging refund app month nobody replying
2,costly premium version approx indian rupees pe...
3,used keep organized updates made mess things c...
4,dan birthday oct


In [25]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preprocessed_reviews["review"])

In [26]:
clust_kmeans = KMeans(n_clusters=5, random_state=500)
pred_labels = clust_kmeans.fit_predict(tfidf_matrix)

In [27]:
categories = pred_labels.tolist()
preprocessed_reviews["category"] = categories

In [28]:
terms = vectorizer.get_feature_names_out()
print(terms)

['aaah' 'aak' 'aap' ... 'گزینه' 'ইন' 'লগ']


In [29]:
topic_terms_list = []

for cluster in range(clust_kmeans.n_clusters):
    
    cluster_indices = [i for i, label in enumerate(categories) if label == cluster]

    cluster_tfidf_sum = tfidf_matrix[cluster_indices].sum(axis=0)
    cluster_term_freq = np.asarray(cluster_tfidf_sum).ravel()

    top_term_index = cluster_term_freq.argsort()[::-1][0]
    
    topic_terms_list.append(
        {
            "category": cluster,
            "term": terms[top_term_index],
            "frequency": cluster_term_freq[top_term_index],
        }
    )


topic_terms = pd.DataFrame(topic_terms_list)

print(topic_terms)


   category     term   frequency
0         0  version   68.212842
1         1      ads   48.150007
2         2     work   71.265038
3         3      app  183.228187
4         4     good   54.423558
